In [9]:
import pandas as pd
import re

books_bx = pd.read_csv('D:/documents/Smart_Library/data/raw/Books.csv', encoding='latin-1', on_bad_lines='skip', dtype={'Year-Of-Publication': str})
ratings_bx = pd.read_csv('D:/documents/Smart_Library/data/raw/Ratings.csv', encoding='latin-1', on_bad_lines='skip')
books_15k = pd.read_csv('D:/documents/Smart_Library/data/raw/google_books_dataset.csv')

print(f"CHECKPOINT 1-  raw books_15k: {len(books_15k)}")

CHECKPOINT 1-  raw books_15k: 15147


In [10]:
print(books_15k.columns.to_list())
print(books_bx.columns.to_list())
print(ratings_bx.columns.to_list())

['book_id', 'title', 'subtitle', 'authors', 'publisher', 'published_date', 'description', 'page_count', 'categories', 'average_rating', 'ratings_count', 'language', 'preview_link', 'info_link', 'isbn_13', 'isbn_10', 'list_price', 'currency', 'buyable', 'search_category', 'thumbnail']
['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']
['User-ID', 'ISBN', 'Book-Rating']


In [11]:
books_15k['isbn_13'] = books_15k['isbn_13'].apply(lambda x:str(int(x)) if pd.notna(x) else None )
books_15k['isbn_10'] = books_15k['isbn_10'].apply(lambda x:str(x).strip() if pd.notna(x) else None )

print(f"Checkpoint 2, After isbn fix: {len(books_15k)}")

Checkpoint 2, After isbn fix: 15147


In [12]:
def isbn10_to_isbn13(isbn10):
    isbn10 = str(isbn10).strip().replace('-', '')
    if len(isbn10)!=10:
        return None
    core = '978' + isbn10[:9]
    total= sum(int(d)*(1 if i%2 == 0 else 3) for i,d in enumerate(core) if d.isdigit())
    check = (10 - (total%10)) % 10
    return core+ str(check)


def to_isbn13(isbn):
    isbn  = str(isbn).strip().replace('-','')
    if len(isbn) == 13:
        return isbn
    elif len(isbn) == 10:
        return isbn10_to_isbn13(isbn)
    else:
        return None

books_bx['isbn13'] = books_bx['ISBN'].apply(to_isbn13)
books_15k = books_15k.rename(columns={'isbn_13': 'isbn13', 'isbn_10':'isbn10'})

assert isbn10_to_isbn13('0439554934') == '9780439554930'
print("Checksum verified")

print("Converted ssuccessfully (books crossing)", books_bx['isbn13'].notna().sum())
print("Failed (books crossing)", books_bx['isbn13'].isna().sum())

print(f"Checkpoint 3 after rename: {len(books_15k)}")

Checksum verified
Converted ssuccessfully (books crossing) 271360
Failed (books crossing) 0
Checkpoint 3 after rename: 15147


In [13]:
def normalize_text(s):
    if pd.isna(s):
        return ''
    s =str(s).lower().strip()
    s= re.sub(r'[^\w\s]','',s)
    s =re.sub(r'\s+', ' ', s)
    return s

books_bx['title_norm'] = books_bx['Book-Title'].apply(normalize_text)
books_bx['author_norm'] = books_bx['Book-Author'].apply(normalize_text)
books_15k['title_norm'] = books_15k['title'].apply(normalize_text)
books_15k['author_norm'] = books_15k['authors'].apply(normalize_text)

print(f"Checkpoint 4 - after normalization: {len (books_15k)}")


Checkpoint 4 - after normalization: 15147


In [14]:
def is_contaminated(row):
    author = str(row['authors']).lower() if pd.notna(row['authors']) else ''
    publisher = str(row['publisher']).lower() if pd.notna(row['publisher']) else ''
    category = str(row['search_category']).lower() if pd.notna(row['search_category']) else ''
    category_words = set(category.split())
    if not category_words:
        return False
    if publisher in category_words:
        return True
    if any(f", {w}" in author for w in category_words):
        return True
    return False

junk_mask = books_15k['authors'].isna() & books_15k['isbn13'].isna() & books_15k['description'].isna()
contaminated_mask = books_15k.apply(is_contaminated, axis=1)

print(f"checkpoint 5a before removing: {len(books_15k)} row")
print(f"Junk found: {junk_mask.sum()}, Contaminated found: {contaminated_mask.sum()}")

books_15k = books_15k[~junk_mask & ~contaminated_mask].copy()
books_15k = books_15k.drop_duplicates(subset=['title_norm','isbn13'], keep='first').copy()
books_15k = books_15k.reset_index(drop=True)

print(f"Checkpount 5b ; final: {len(books_15k)} rows")
    
   

checkpoint 5a before removing: 15147 row
Junk found: 3041, Contaminated found: 11
Checkpount 5b ; final: 11385 rows


In [15]:
books_15k.head(3)


,book_id,title,subtitle,authors,publisher,published_date,description,page_count,categories,average_rating,...,info_link,isbn13,isbn10,list_price,currency,buyable,search_category,thumbnail,title_norm,author_norm
0,4fXUDAAAQBAJ,The Bestseller Code,Anatomy of the Blockbuster Novel,"Jodie Archer, Matthew L. Jockers",Macmillan,2016-09-20,"""What if there was an algorithm that could pre...",253.0,Business & Economics,NaN,...,http://books.google.com/books?id=4fXUDAAAQBAJ&...,9781250088277,1250088275,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=4fXUD...,the bestseller code,jodie archer matthew l jockers
1,yIVuDwAAQBAJ,Bestseller,A Century of America's Favorite Books,Robert McParland,Bloomsbury Publishing PLC,2018-12-15,Whether curled up on a sofa with a good myster...,335.0,Literary Criticism,NaN,...,https://play.google.com/store/books/details?id...,9781538110003,1538110008,40.5,USD,True,fiction bestsellers,http://books.google.com/books/content?id=yIVuD...,bestseller,robert mcparland
2,2JHXwAEACAAJ,Bestsellers: Popular Fiction since 1900,NaN,C. Bloom,Palgrave Macmillan,2002-07-09,This guide and reference work of all of the be...,306.0,Literary Criticism,NaN,...,http://books.google.com/books?id=2JHXwAEACAAJ&...,9780333687420,0333687426,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=2JHXw...,bestsellers popular fiction since 1900,c bloom


In [16]:
books_15k['full_title'] = books_15k.apply(lambda r: f"{r['title']}: {r['subtitle']}" if pd.notna(r['subtitle']) else r['title'], axis =1 )

Merge on ISBN

In [17]:
merged = pd.merge(
    books_bx, books_15k,
    on='isbn13', how='outer',
    suffixes=('_bx','_gb'),
    indicator= True
)

print(merged['_merge'].value_counts())

_merge
left_only     271206
right_only     11232
both             154
Name: count, dtype: int64


In [18]:
print("books_bx isbn13 dtype:", books_bx['isbn13'].dtype)
print("books_15k isbn13 dtype:", books_15k['isbn13'].dtype)

bx_isbns = set(books_bx['isbn13'].dropna())
gb_isbns = set(books_15k['isbn13'].dropna())
overlap = bx_isbns & gb_isbns
print(f"Direct set overlap:{len(overlap)}")

print(repr(books_bx['isbn13'].dropna().iloc[0]))
print(repr(books_15k['isbn13'].dropna().iloc[0]))

books_bx isbn13 dtype: str
books_15k isbn13 dtype: str
Direct set overlap:153
'9780195153446'
'9781250088277'


In [19]:
print(books_bx['isbn13'].apply(type).value_counts())

isbn13
<class 'str'>    271360
Name: count, dtype: int64


In [20]:
books_bx.head(2)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,isbn13,title_norm,author_norm
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,9780195153446,classical mythology,mark p o morford
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,9780002005012,clara callan,richard bruce wright


In [21]:
matched_isbns =  set(books_bx['isbn13'].dropna()) & set(books_15k['isbn13'].dropna())

unmatched_bx = books_bx[~books_bx['isbn13'].isin(matched_isbns)].copy()
unmatched_gb = books_15k[~books_15k['isbn13'].isin(matched_isbns)].copy()

fallback = pd.merge(
    unmatched_bx,unmatched_gb,
    on = ['title_norm', 'author_norm'], how='inner',
    suffixes=('_bx','_gb'),
)

print(f"Fallback matched {len(fallback)} additional books")

Fallback matched 822 additional books


In [22]:
#fixed ghis part where the isbn of thensame book was different, just filtered them, not dropped to calcultae the actual book count

unmatched_bx_deduped = unmatched_bx.drop_duplicates(subset=['title_norm','author_norm'],keep='first')

print(f"unmatched_bx before dedup:{len(unmatched_bx)}")
print(f"unmatched_bx after dedup:{len(unmatched_bx_deduped)}")

fallback = pd.merge(
    unmatched_bx_deduped, unmatched_gb,
    on=['title_norm','author_norm'], how='inner',
    suffixes=('_bx','_gb')
)

print(f"Fallback matched (deduped): {len(fallback)} additional books")

unmatched_bx before dedup:271206
unmatched_bx after dedup:245738
Fallback matched (deduped): 309 additional books


In [23]:
def build_catalog(df, isbn_col, title_col, author_col, desc_col=None, cat_col=None, cover_col=None, source=''):
    return pd.DataFrame({
        'isbn13': df[isbn_col],
        'title': df[title_col],
        'author': df[author_col],
        'description': df[desc_col] if desc_col else None,
        'categories': df[cat_col] if cat_col else None,
        'cover_url': df[cover_col] if cover_col else None,
        'source': source 
    })

matched_bx = books_bx[books_bx['isbn13'].isin(matched_isbns)]
matched_gb = books_15k[books_15k['isbn13'].isin(matched_isbns)]

both_merged = pd.merge(matched_bx, matched_gb, on='isbn13', suffixes=('_bx','_gb'))

catalog_both = build_catalog(both_merged,'isbn13', 'title', 'Book-Author', 'description','categories','thumbnail', 'isbn_match')
catalog_fallback = build_catalog(fallback,'isbn13_gb', 'title', 'Book-Author', 'description','categories','thumbnail', 'title_match')


bx_matched_isbns = set(matched_bx['isbn13']) | set(fallback['isbn13_bx'].dropna())
gb_matched_isbns = set(matched_gb['isbn13']) | set(fallback['isbn13_gb'].dropna())


bx_only = books_bx[~books_bx['isbn13'].isin(bx_matched_isbns)]
gb_only = books_15k[~books_15k['isbn13'].isin(gb_matched_isbns)]

catalog_bx_only = build_catalog(bx_only, 'isbn13','Book-Title','Book-Author', source='book_crossing_only')
catalog_gb_only = build_catalog(gb_only, 'isbn13','title','authors','description','categories','thumbnail','google_books_only')


final_catalog = pd.concat([catalog_both,catalog_fallback,catalog_bx_only,catalog_gb_only], ignore_index=True)

print(f"Final unified catalog: {len(final_catalog)} books")
print(final_catalog['source'].value_counts())


Final unified catalog: 282396 books
source
book_crossing_only    270972
google_books_only      10961
title_match              309
isbn_match               154
Name: count, dtype: int64


In [24]:
print(final_catalog[final_catalog['source'] == 'isbn_match'][['title','author']].head(5))

              title         author
0           Beloved  Toni Morrison
1   A Painted House   John Grisham
2  Heart of the Sea   Nora Roberts
3       The Partner   John Grisham
4        Neverwhere    Neil Gaiman


Checking for errors

In [25]:
dup_bx = books_bx[books_bx.duplicated(subset=['title_norm','author_norm'], keep=False)]
print(f"Book_Crossing having duplicate rows of author+title: {len(dup_bx)}")

dup_gb = books_15k[books_15k.duplicated(subset=['title_norm','author_norm'], keep=False)]
print(f"Google books having duplicate rows of author+title: {len(dup_gb)}")

print(f"\n Rows removed from bx: {len(books_bx)-len(bx_only)}")
print(f"Unique  ISBN targeted for removal: {len(bx_matched_isbns)}")

Book_Crossing having duplicate rows of author+title: 44855
Google books having duplicate rows of author+title: 521

 Rows removed from bx: 388
Unique  ISBN targeted for removal: 385


Checked and fixed above in the section of fallback

In [26]:
final_catalog.to_parquet('D:/documents/Smart_Library/data/processed/books_clean.parquet',engine='fastparquet', index=False)
print(f"Saved {len(final_catalog)} books to book_clean.parquet")

Saved 282396 books to book_clean.parquet


In [34]:
final_catalog.loc[final_catalog['source'] == 'book_crossing_only', 'cover_url'] = 'https://covers.openlibrary.org/b/isbn/' + bx_only['isbn13'].astype(str) + '-L.jpg'
final_catalog = pd.concat([final_catalog, openlibrary_df], ignore_index=True)

print(f"Tital books now: {len(final_catalog)}")
print(f"Rows still missing a cover: {final_catalog['cover_url'].isna().sum()}")
print(final_catalog['source'].value_counts())

final_catalog.to_parquet('D:/documents/Smart_Library/data/processed/books_clean.parquet',engine='fastparquet', index=False)
print(f"saved {len(final_catalog)} books to book_clean.parquet")

Tital books now: 286596
Rows still missing a cover: 1306
source
book_crossing_only       270972
google_books_only         10961
open_library_academic      4200
title_match                 309
isbn_match                  154
Name: count, dtype: int64
saved 286596 books to book_clean.parquet


In [36]:
before = len(final_catalog)

final_catalog = final_catalog.drop_duplicates(keep='first').reset_index(drop =True)
print(f"Before: {before}, After : {len(final_catalog)} (removed {before - len(final_catalog)})")
print(final_catalog['source'].value_counts())

final_catalog.to_parquet('D:/documents/Smart_Library/data/processed/books_clean.parquet',engine='fastparquet', index=False)
print(f"saved {len(final_catalog)} books to book_clean.parquet")

Before: 284495, After : 284495 (removed 0)
source
book_crossing_only       270972
google_books_only         10961
open_library_academic      2100
title_match                 309
isbn_match                  153
Name: count, dtype: int64
saved 284495 books to book_clean.parquet


Phase 1 completed, moving to phase 1.8

In [ ]:
import requests
import time
import pandas as pd

subjects = ['computer_science','mathematics','physics','engineering','business','philosophy','psychology']
headers = {'User-Agent' : 'SmartLibraryStudentProject/1.0'}

all_records = []
failed_requests = []

for subject in subjects:
    print(f"Fetching subject : {subject}")
    offset = 0
    limit = 100

    while offset < 300:
        url = f"https://openlibrary.org/subjects/{subject}.json?limit={limit}&offset={offset}"

        success = False
        for attempt in range(3):
            try: 
                response = requests.get(url, headers =headers, timeout=10)
                if response.status_code == 200:
                    success = True
                    break
                else:
                    print(f" Status {response.status_code} at offset {offset}, attempt {attempt+1}")   
            except requests.exceptions.RequestException as e:
                print(f"  Connection failed as offset {offset}, attempt {attempt+1}: {type(e).__name__}")
                time.sleep(2)
        if not success:
            print(f" Giving up on subejct {subject} at offset {offset} after 3 attempts")
            failed_requests.append((subject, offset))
            offset += limit
            continue

        data = response.json()
        works = data.get('works',[])
        if not  works:
            break
        
        for w in works:
            all_records.append({
                'isbn13': None,
                'title': w.get('title'),
                'author': ','.join(a['name']  for a in w.get('authors', [])),
                'description' : None,
                'categories': subject,
                'cover_url' : f"https://covers.openlibrary.org/b/id/{w['cover_id']}-M.jpg" if w.get('cover_id') else None,
                'source' : 'open_library_academic'
            })

        offset += limit
        time.sleep(0.5)

openlibrary_df = pd.DataFrame(all_records)
print(f"\nTotaal academic book pulled: {len(openlibrary_df)}")
print(openlibrary_df['categories'].value_counts())
print(openlibrary_df[['title','author','categories']].head(5))

Fetching subject : computer_science
Fetching subject : mathematics
Fetching subject : physics
Fetching subject : engineering
Fetching subject : business
Fetching subject : philosophy
Fetching subject : psychology

Totaal academic book pulled: 2100
categories
computer_science    300
mathematics         300
physics             300
engineering         300
business            300
philosophy          300
psychology          300
Name: count, dtype: int64
                                               title  \
0  Advances in Computers, Volume 49 (Advances in ...   
1  Job scheduling strategies for parallel processing   
2           Transactions on Engineering Technologies   
3                                Hacking for Dummies   
4  Artificial Intelligence Applications and Innov...   

                                              author        categories  
0             Marvin V. Zelkowitz,Marshall C. Yovits  computer_science  
1  JSSPP'99 (1999 San Juan, P.R.),IPPS/SPDP '98 W...  computer_s

Phasee 2, normalaization of the rating

In [37]:
print(ratings_bx['Book-Rating'].describe())
print(ratings_bx['Book-Rating'].value_counts().sort_index())

count    1.149780e+06
mean     2.866950e+00
std      3.854184e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      7.000000e+00
max      1.000000e+01
Name: Book-Rating, dtype: float64
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64


In [38]:
explicit = ratings_bx[ratings_bx['Book-Rating']>0].copy()
implicit = ratings_bx[ratings_bx['Book-Rating']==0].copy()

print(f"Explicit rating (real scores): {len(explicit)}")
print(f"Implicit rating (no scores given): {len(implicit)}")


Explicit rating (real scores): 433671
Implicit rating (no scores given): 716109


In [43]:

explicit = explicit.copy()
explicit['rating_normalized'] =pd.to_numeric(explicit['Book-Rating'], errors= 'coerce')
explicit['rating_normalized'] = (explicit['Book-Rating']-1)/9*5
print(
    explicit[['Book-Rating', 'rating_normalized']].drop_duplicates().sort_values('Book-Rating'))

     Book-Rating  rating_normalized
241            1           0.000000
370            2           0.555556
3              3           1.111111
55             4           1.666667
1              5           2.222222
4              6           2.777778
8              7           3.333333
6              8           3.888889
16             9           4.444444
9             10           5.000000


In [45]:
implicit['event_type'] = 'implicit_interaction'
explicit['event_type'] = 'explicit_rating'

ratings_clean = pd.concat([explicit, implicit],ignore_index= True)
print(ratings_clean['event_type'].value_counts())

event_type
implicit_interaction    716109
explicit_rating         433671
Name: count, dtype: int64


In [47]:
ratings_clean.to_parquet('D:/documents/Smart_Library/data/processed/ratings_clean.parquet', index=False)
print(f"Saved {len(ratings_clean)} rating records")

Saved 1149780 rating records
